In [ ]:
import re
import sys
import csv
import numpy as np
from pathlib import Path as ph
import matplotlib.pyplot as plt

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
def process_visualize_file_components():
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    attentions = ['mha', 'moh', 'gqa', 'swh', 'lda', 'rfa']
    networks = ['mlp', 'moe', 'lor', 'swi', 'lin', 'ggl'] 
    n_layers = [4,8,16]

    # Extract header and data
    header = statistic[0]
    header[0] = header[0].lstrip('\ufeff').lstrip('\ufeff') # Remove BOM if present
    data = statistic[1:]

    # Find columns index
    
    x_col_a = header.index("baby's brain")
    w_col_b = header.index("c_device")
    w_col_c = header.index("n_layers")
    w_col_d = header.index("c_attention")
    w_col_e = header.index("c_network")
    w_col_f = header.index("inference_quality_execution")

    fig, axes = plt.subplots(len(n_layers), 2, figsize=(24, 8 * len(n_layers)))  # Removed sharex=True
    if len(n_layers) == 1:
        axes = [axes]

    for idx, n_layer in enumerate(n_layers):
        combos = {}
        combos_finetuned = {}
        for row in data:
            # Filter data
            if str(row[w_col_b]) == 'gpu' and int(row[w_col_c]) == n_layer:
                att = row[w_col_d]
                net = row[w_col_e]
                combo = float(row[w_col_f].replace("/100", ""))
                if not str(row[x_col_a]).endswith('_finetuned'):
                    combos[(att, net)] = combo
                else:
                    combos_finetuned[(att, net)] = combo

        # Build existence matrix
        matrix = []
        matrix_finetuned = []
        for net in networks:
            elements = []
            elements_finetuned = []
            for att in attentions:
                elements.append(combos.get((att, net), np.nan))
                elements_finetuned.append(combos_finetuned.get((att, net), np.nan))
            matrix.append(elements)
            matrix_finetuned.append(elements_finetuned)

        # Plot
        ax = axes[idx][0] if len(n_layers) > 1 else axes[0]
        im = ax.imshow(matrix, cmap='Greens', aspect='auto')
        ax.set_xticks(range(len(attentions)))
        ax.set_xticklabels(attentions, rotation=45)
        ax.set_yticks(range(len(networks)))
        ax.set_yticklabels(networks)
        ax.set_xlabel('Attention Mechanism')
        ax.set_ylabel('Network Mechanism')
        ax.set_title(f'Combinations for n_layers={n_layer}')
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        plt.colorbar(im, label='Quality')

        # Plot finetuned
        ax = axes[idx][1] if len(n_layers) > 1 else axes[1]
        im_finetuned = ax.imshow(matrix_finetuned, cmap='Greens', aspect='auto')
        ax.set_xticks(range(len(attentions)))
        ax.set_xticklabels(attentions, rotation=45)
        ax.set_yticks(range(len(networks)))
        ax.set_yticklabels(networks)
        ax.set_xlabel('Attention Mechanism')
        ax.set_ylabel('Network Mechanism')
        ax.set_title(f'Combinations for n_layers={n_layer} (Finetuned)')
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        plt.colorbar(im_finetuned, label='Quality')

    plt.tight_layout()
    plt.savefig(f"{statistics_path}/statistic_experiments_matrixes.png", bbox_inches='tight')
    plt.show()

In [ ]:
process_visualize_file_components()